In [ ]:
import numpy as np

from detectors import RegionalDriftDetector, DriftDetector
from mmdew.bucket_stream2 import BucketStream
from mmdew.mmd import MMD

class MMDEWAdapter(DriftDetector):
    def __init__(self, gamma, alpha=.1):
        """
        :param gamma: The scale of the data
        :param alpha: alpha value for the hypothesis test
      
        """
        self.gamma=gamma
        self.alpha = alpha
        self.logger = None
        self.detector = BucketStream(gamma=self.gamma, compress=True, alpha=self.alpha)
        self.element_count = 0
        super(MMDEWAdapter, self).__init__()

    def name(self) -> str:
        return "MMDEW" 

    def parameter_str(self) -> str:
        return r"$\alpha = {}$".format(self.alpha)

    def pre_train(self, data):
        # hier können wir estimate_gamma ausführen
        self.gamma = MMD.estimate_gamma(data)
        #print(f"gamma: {self.gamma}")
        self.detector = BucketStream(gamma=self.gamma, compress=True, alpha=self.alpha)
    

    def add_element(self, input_value):
        """
        Add the new element and also perform change detection
        :param input_value: The new observation
        :return:
        """

        self.element_count+=1
        self.detected_cp = False
        prev_cps = len(self.detector.get_changepoints())
        self.detector.insert(input_value[0])
        if len(self.detector.get_changepoints()) > prev_cps:
            self.delay = self.element_count - self.detector.get_changepoints()[-1]
            self.detected_cp = True
#            print("Detected")

    def detected_change(self):
        return self.detected_cp
    
    def metric(self):
        return 0

 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import dists



In [ ]:
rng = np.random.default_rng()

In [ ]:
n_post_change = 10000
d=5
num_runs = 10
thresholds = thresholds = np.linspace(1e-2,99e-2,endpoint=True,num=10)

In [ ]:
for dist in [dists.DistributionType.LAPLACE,dists.DistributionType.UNIFORM,dists.DistributionType.MIXED]:
    res_arl = pd.DataFrame()
    for _ in range(num_runs):
        for t in thresholds:
            Q = dists._get_distribution(dist.value,n_post_change,d)

            adapter = MMDEWAdapter(gamma=0,alpha=t)
            adapter.pre_train(data=Q)

            arl = []


            for i, x in enumerate(Q):
                adapter.add_element(np.asarray([x]))
                if adapter.detected_change():
                    arl += [i]

            arl = np.diff(np.array(arl))
            if len(arl) >= 1:
                res_arl = pd.concat((res_arl,pd.DataFrame({"threshold" : [t]*len(arl), "arl":arl})))
            else:
                res_arl = pd.concat((res_arl,pd.DataFrame({"threshold" : [t], "arl":np.nan})))
    res_arl.to_csv(f"results/arl/mmdew_{dist}.csv")